<a href="https://colab.research.google.com/github/priyanshisharma919346-del/Priya/blob/main/Bitcoin_Market_Sentiment_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
data = pd.read_csv("/content/historical_data.csv", low_memory=False)
data1 = pd.read_csv("/content/fear_greed_index.csv", low_memory=False)

data = data.drop(columns=["Account", "Transaction Hash"])

In [ ]:
data.head()

In [ ]:
data['date'] = pd.to_datetime(data['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.date
data1['date'] = pd.to_datetime(data1['date']).dt.date
merged_data = pd.merge(data, data1, on='date', how='inner')
merged_data.head()

In [ ]:
X = merged_data.drop("Closed PnL",axis = 1)
Y = merged_data["Closed PnL"]

In [ ]:
sentiment_map = {
    'Extreme Fear': 1,
    'Fear': 2,
    'Neutral': 3,
    'Greed': 4,
    'Extreme Greed': 5
}

X_data = merged_data[['Execution Price', 'Size Tokens', 'Size USD', 'value', 'classification']]

X_data['classification'] = X_data['classification'].map(sentiment_map)

Y_data = merged_data["Closed PnL"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
X_train = model.fit(X_train_scaled, Y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test_scaled)
print(f"Mean Absolute Error",mean_absolute_error(Y_test, y_pred))
print(f"Mean Squared Error", mean_squared_error(Y_test, y_pred))
print(f"R2 Score", r2_score(Y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

importances = model.feature_importances_
features = X_data.columns

plt.figure(figsize=(10, 5))
plt.barh(features, importances, color='skyblue')
plt.xlabel('Importance Score')
plt.title('which feature change the profit of Row?')
plt.show()